# Week 12 Improving Model Performance
> by Lei Ding, Updated Apr. 2025

In [1]:
#install.packages('caret')
#install.packages('ipred')
#install.packages('vcd')
#install.packages('adabag')

## 1. 自动参数调整

In [2]:
##### Improving Model Performance -------------------

# load the credit dataset
credit <- read.csv("credit.csv", stringsAsFactors = TRUE)
str(credit)

'data.frame':	1000 obs. of  17 variables:
 $ checking_balance    : Factor w/ 4 levels "1 - 200 DM","< 0 DM",..: 2 1 4 2 2 4 4 1 4 1 ...
 $ months_loan_duration: int  6 48 12 42 24 36 24 36 12 30 ...
 $ credit_history      : Factor w/ 5 levels "critical","good",..: 1 2 1 2 4 2 2 2 2 1 ...
 $ purpose             : Factor w/ 6 levels "business","car",..: 5 5 4 5 2 4 5 2 5 2 ...
 $ amount              : int  1169 5951 2096 7882 4870 9055 2835 6948 3059 5234 ...
 $ savings_balance     : Factor w/ 5 levels "100 - 500 DM",..: 5 3 3 3 3 5 2 3 4 3 ...
 $ employment_duration : Factor w/ 5 levels "1 - 4 years",..: 4 1 2 2 1 1 4 1 2 5 ...
 $ percent_of_income   : int  4 2 2 2 3 2 3 2 2 4 ...
 $ years_at_residence  : int  4 2 3 4 4 4 4 2 4 2 ...
 $ age                 : int  67 22 49 45 53 35 53 35 61 28 ...
 $ other_credit        : Factor w/ 3 levels "bank","none",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ housing             : Factor w/ 3 levels "other","own",..: 2 2 2 1 1 1 2 3 2 2 ...
 $ existing_loans_cou

In [3]:
library(caret)

## Creating a simple tuned model ----
# automated parameter tuning of C5.0 decision tree 
RNGversion("3.5.2") # use an older random number generator to match the book
set.seed(300)
m <- train(default ~ ., data = credit, method = "C5.0")

# summary of tuning results
m

Loading required package: ggplot2

Loading required package: lattice

Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
“non-uniform 'Rounding' sampler used”


C5.0 

1000 samples
  16 predictor
   2 classes: 'no', 'yes' 

No pre-processing
Resampling: Bootstrapped (25 reps) 
Summary of sample sizes: 1000, 1000, 1000, 1000, 1000, 1000, ... 
Resampling results across tuning parameters:

  model  winnow  trials  Accuracy   Kappa    
  rules  FALSE    1      0.6936796  0.2736827
  rules  FALSE   10      0.7190566  0.3303172
  rules  FALSE   20      0.7288049  0.3421719
  rules   TRUE    1      0.6958695  0.2768299
  rules   TRUE   10      0.7175839  0.3224745
  rules   TRUE   20      0.7292436  0.3444603
  tree   FALSE    1      0.6916674  0.2559863
  tree   FALSE   10      0.7316373  0.3165534
  tree   FALSE   20      0.7382427  0.3305229
  tree    TRUE    1      0.6963560  0.2681042
  tree    TRUE   10      0.7271986  0.3047028
  tree    TRUE   20      0.7368198  0.3263065

Accuracy was used to select the optimal model using the largest value.
The final values used for the model were trials = 20, model = tree and winnow
 = FALSE.

In [4]:
# apply the best C5.0 candidate model to make predictions
p <- predict(m, credit)
table(p, credit$default)

     
p      no yes
  no  700   2
  yes   0 298

In [5]:
# obtain predicted classes
head(predict(m, credit, type = "raw"))

[1] no  yes no  no  yes no 
Levels: no yes

In [6]:
# obtain predicted probabilities
head(predict(m, credit, type = "prob"))

,no,yes
,<dbl>,<dbl>
1,0.86968459,0.13031541
2,0.05202096,0.94797904
3,0.90548588,0.09451412
4,0.77601287,0.22398713
5,0.26854834,0.73145166
6,0.87450765,0.12549235


In [7]:
## Customizing the tuning process ----
# use trainControl() to alter resampling strategy
ctrl <- trainControl(method = "cv", number = 10, selectionFunction = "oneSE")

# use expand.grid() to create grid of tuning parameters
grid <- expand.grid(model = "tree", trials = c(1, 5, 10, 15, 20, 25, 30, 35), winnow = FALSE)

# look at the result of expand.grid()
grid

model,trials,winnow
<fct>,<dbl>,<lgl>
tree,1,FALSE
tree,5,FALSE
tree,10,FALSE
tree,15,FALSE
tree,20,FALSE
tree,25,FALSE
tree,30,FALSE
tree,35,FALSE


In [8]:
# customize train() with the control list and grid of parameters 
RNGversion("3.5.2") # use an older random number generator to match the book
set.seed(300)
m <- train(default ~ ., data = credit, method = "C5.0", metric = "Kappa", trControl = ctrl, tuneGrid = grid)
m

Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
“non-uniform 'Rounding' sampler used”


C5.0 

1000 samples
  16 predictor
   2 classes: 'no', 'yes' 

No pre-processing
Resampling: Cross-Validated (10 fold) 
Summary of sample sizes: 900, 900, 900, 900, 900, 900, ... 
Resampling results across tuning parameters:

  trials  Accuracy  Kappa    
   1      0.710     0.2649945
   5      0.708     0.2579961
  10      0.740     0.3361958
  15      0.745     0.3472762
  20      0.746     0.3515305
  25      0.753     0.3664771
  30      0.741     0.3404058
  35      0.752     0.3623037

Tuning parameter 'model' was held constant at a value of tree
Tuning
 parameter 'winnow' was held constant at a value of FALSE
Kappa was used to select the optimal model using  the one SE rule.
The final values used for the model were trials = 15, model = tree and winnow
 = FALSE.

## 2. 装袋

In [9]:
## Bagging ----
# Using the ipred bagged decision trees
library(ipred)
RNGversion("3.5.2") # use an older random number generator to match the book
set.seed(300)
mybag <- bagging(default ~ ., data = credit, nbagg = 25)
credit_pred <- predict(mybag, credit)
table(credit_pred, credit$default)

Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
“non-uniform 'Rounding' sampler used”


           
credit_pred  no yes
        no  699   2
        yes   1 298

In [10]:
# estimate performance of ipred bagged trees
library(caret)
RNGversion("3.5.2") # use an older random number generator to match the book
set.seed(300)
ctrl <- trainControl(method = "cv", number = 10)
train(default ~ ., data = credit, method = "treebag", trControl = ctrl)

Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
“non-uniform 'Rounding' sampler used”


Bagged CART 

1000 samples
  16 predictor
   2 classes: 'no', 'yes' 

No pre-processing
Resampling: Cross-Validated (10 fold) 
Summary of sample sizes: 900, 900, 900, 900, 900, 900, ... 
Resampling results:

  Accuracy  Kappa    
  0.738     0.3289352


## 3. 随机森林

In [11]:
## Random Forests ----
# random forest with default settings
library(randomForest)
RNGversion("3.5.2") # use an older random number generator to match the book
set.seed(300)
rf <- randomForest(default ~ ., data = credit)
rf

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: ‘randomForest’


The following object is masked from ‘package:ggplot2’:

    margin


Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
“non-uniform 'Rounding' sampler used”



Call:
 randomForest(formula = default ~ ., data = credit) 
               Type of random forest: classification
                     Number of trees: 500
No. of variables tried at each split: 4

        OOB estimate of  error rate: 23.3%
Confusion matrix:
     no yes class.error
no  638  62  0.08857143
yes 171 129  0.57000000

In [12]:
# calculate kappa on the out-of-bag estimate
library(vcd)
Kappa(rf$confusion[1:2,1:2])

Loading required package: grid



           value     ASE     z  Pr(>|z|)
Unweighted 0.381 0.03215 11.85 2.197e-32
Weighted   0.381 0.03215 11.85 2.197e-32

## 4. 提升

In [13]:
## Boosting ----

## Using C5.0 Decision Tree (not shown in book)
library(C50)
m_c50_bst <- C5.0(default ~ ., data = credit, trials = 100)
credit_pred <- predict(m_c50_bst, credit)
table(credit_pred, credit$default)

           
credit_pred  no yes
        no  700   4
        yes   0 296

In [14]:
## Using AdaBoost.M1
library(adabag)

# create a Adaboost.M1 model
RNGversion("3.5.2") # use an older random number generator to match the book
set.seed(300)
m_adaboost <- boosting(default ~ ., data = credit)
p_adaboost <- predict(m_adaboost, credit)
head(p_adaboost$class)

Loading required package: rpart

Loading required package: foreach

Loading required package: doParallel

Loading required package: iterators

Loading required package: parallel


Attaching package: ‘adabag’


The following object is masked from ‘package:ipred’:

    bagging


Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
“non-uniform 'Rounding' sampler used”


[1] "no"  "yes" "no"  "no"  "yes" "no"

In [15]:
p_adaboost$confusion

               Observed Class
Predicted Class  no yes
            no  700   0
            yes   0 300

In [16]:
# create and evaluate an Adaboost.M1 model using 10-fold-CV
RNGversion("3.5.2") # use an older random number generator to match the book
set.seed(300)
adaboost_cv <- boosting.cv(default ~ ., data = credit)
adaboost_cv$confusion

Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
“non-uniform 'Rounding' sampler used”


i:  1 Mon Apr 21 10:53:36 2025 
i:  2 Mon Apr 21 10:53:41 2025 
i:  3 Mon Apr 21 10:53:46 2025 
i:  4 Mon Apr 21 10:53:50 2025 
i:  5 Mon Apr 21 10:53:55 2025 
i:  6 Mon Apr 21 10:54:00 2025 
i:  7 Mon Apr 21 10:54:05 2025 
i:  8 Mon Apr 21 10:54:09 2025 
i:  9 Mon Apr 21 10:54:14 2025 
i:  10 Mon Apr 21 10:54:19 2025 


               Observed Class
Predicted Class  no yes
            no  593 148
            yes 107 152

In [17]:
# calculate kappa
library(vcd)
Kappa(adaboost_cv$confusion)

            value    ASE     z  Pr(>|z|)
Unweighted 0.3682 0.0322 11.43 2.819e-30
Weighted   0.3682 0.0322 11.43 2.819e-30